In [3]:
!pip install -q openai-whisper
!pip install -q transformers accelerate torch sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 44.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import whisper
import torch
import json
from transformers import pipeline
from google.colab import files

In [5]:
uploaded = files.upload()

Saving voice.mp3 to voice.mp3


In [6]:
model = whisper.load_model("small")

100%|████████████████████████████████████████| 461M/461M [00:01<00:00, 306MiB/s]


In [7]:
audio_file = list(uploaded.keys())[0]

result = model.transcribe(
    audio_file,
    language="hi",
    task="translate",
    fp16=False
)

transcript = result["text"]

print(transcript)

 Good afternoon, thank you for calling Sunrise Hotel. My name is Raul. How may I assist you today? I want to book a room for this weekend. Sure sir, may I know the check-in and check-out dates? I want to check-in on Saturday and check-out on Monday morning. Alright, how many guests will be staying? Two adults. Okay, we currently have Deluxe rooms and Executive rooms available. Which one would you prefer? Can you tell me the price difference? Yes sir, the Deluxe room costs 3500 per night and the Executive room costs 5000 per night including complimentary breakfast. Okay, I think I will go with the Deluxe room. Sure sir, would you like a room with a city view? Yes, that would be great. Perfect, may I have your full name please? My name is Aman Sharma. Thank you Mr Aman, could you also provide your contact number? Yes, it is 9876543210. Thank you, your room has been successfully reserved for two nights in Deluxe city view room. Great, is breakfast included? Breakfast is not included in th

In [8]:
lines = transcript.split('.')

formatted_transcript = ""

for i, line in enumerate(lines):
    line = line.strip()

    if line:
        speaker = "Agent" if i % 2 == 0 else "Customer"
        formatted_transcript += f"{speaker}: {line}\n"

print(formatted_transcript)

Agent: Good afternoon, thank you for calling Sunrise Hotel
Customer: My name is Raul
Agent: How may I assist you today? I want to book a room for this weekend
Customer: Sure sir, may I know the check-in and check-out dates? I want to check-in on Saturday and check-out on Monday morning
Agent: Alright, how many guests will be staying? Two adults
Customer: Okay, we currently have Deluxe rooms and Executive rooms available
Agent: Which one would you prefer? Can you tell me the price difference? Yes sir, the Deluxe room costs 3500 per night and the Executive room costs 5000 per night including complimentary breakfast
Customer: Okay, I think I will go with the Deluxe room
Agent: Sure sir, would you like a room with a city view? Yes, that would be great
Customer: Perfect, may I have your full name please? My name is Aman Sharma
Agent: Thank you Mr Aman, could you also provide your contact number? Yes, it is 9876543210
Customer: Thank you, your room has been successfully reserved for two nigh

In [9]:
pipe = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [10]:
prompt = f"""
You are an AI Call Quality Analyst.
Analyze the following customer support conversation
Return STRICT JSON output with:

1. Call_summary
2. Sentiment
3. agent_score_out_of_10
4. Strengths
5. improvement_areas
6. recommended_next_steps

Transcript:
{formatted_transcript}
STRICT JSON ONLY.
"""

In [11]:
response = pipe(
    prompt,
    max_new_tokens=300
)

output = response[0]["generated_text"]

print(output)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an AI Call Quality Analyst.
Analyze the following customer support conversation
Return STRICT JSON output with:

1. Call_summary
2. Sentiment
3. agent_score_out_of_10
4. Strengths
5. improvement_areas
6. recommended_next_steps

Transcript:
Agent: Good afternoon, thank you for calling Sunrise Hotel
Customer: My name is Raul
Agent: How may I assist you today? I want to book a room for this weekend
Customer: Sure sir, may I know the check-in and check-out dates? I want to check-in on Saturday and check-out on Monday morning
Agent: Alright, how many guests will be staying? Two adults
Customer: Okay, we currently have Deluxe rooms and Executive rooms available
Agent: Which one would you prefer? Can you tell me the price difference? Yes sir, the Deluxe room costs 3500 per night and the Executive room costs 5000 per night including complimentary breakfast
Customer: Okay, I think I will go with the Deluxe room
Agent: Sure sir, would you like a room with a city view? Yes, that would be

In [12]:
start = output.find("{")
end = output.rfind("}")+1

json_output = output[start:end]
print(json_output)

{

"call_summary": "Customer Raul is calling to book a Deluxe room for two nights with city view at Sunrise Hotel. Agents recommended the Deluxe room with city view, priced at 3500 per night, and breakfast was optional to be added at Rs 500 per person.",

"sentiment": "Positive",

"agent_score_out_of_10": 9,

"strengths": [

"Provided clear information on room options",

"Helped customer make a decision",

"Efficiently handled the booking process",

"Offered optional services",

"Polite and professional demeanor"

],

"improvement_areas": [

"Could have provided a discount for the room with breakfast",

"Could have informed about early check-in or late check-out options"

],

"recommended_next_steps": [

"Offer breakfast package at a discounted rate",

"Inform about flexible check-in and check-out times"

]

}


In [13]:
data = json.loads(json_output)

with open("report.json", "w") as f:
    json.dump(data, f, indent=4)

print("JSON report saved successfully.")

JSON report saved successfully.


In [19]:
data = json.loads(json_output)

if isinstance(data["strengths"], str):
    data["strengths"] = [data["strengths"]]

if isinstance(data["improvement_areas"], str):
    data["improvement_areas"] = [data["improvement_areas"]]

if isinstance(data["recommended_next_steps"], str):
    data["recommended_next_steps"] = [data["recommended_next_steps"]]

markdown_report = f"""
# AI Voice Call Analysis Report

## Call Summary
{data["call_summary"]}

## Sentiment
{data["sentiment"]}

## Agent Score (out of 10)
{data["agent_score_out_of_10"]}/10

## Strengths
• {" ".join(data["strengths"])}

## Improvement Areas
• {" ".join(data["improvement_areas"])}

## Recommended Next Steps
• {" ".join(data["recommended_next_steps"])}
"""

print(markdown_report)


# AI Voice Call Analysis Report

## Call Summary
Customer Raul is calling to book a Deluxe room for two nights with city view at Sunrise Hotel. Agents recommended the Deluxe room with city view, priced at 3500 per night, and breakfast was optional to be added at Rs 500 per person.

## Sentiment
Positive

## Agent Score (out of 10)
9/10

## Strengths
• Provided clear information on room options Helped customer make a decision Efficiently handled the booking process Offered optional services Polite and professional demeanor

## Improvement Areas
• Could have provided a discount for the room with breakfast Could have informed about early check-in or late check-out options

## Recommended Next Steps
• Offer breakfast package at a discounted rate Inform about flexible check-in and check-out times



In [20]:
with open("report.md","w") as f:
  f.write(markdown_report)
print("Markdown report saved successfully.")

Markdown report saved successfully.
